# 📊 Notebook 01 — Analyse Exploratoire (EDA) — Flipkart Reviews
**Projet : DarijaLens — Analyse de sentiments Darija**  
**Phase 1 : Baseline Flipkart (anglais)**

---
## Objectifs de ce notebook
1. Comprendre la structure et la qualité du dataset Flipkart
2. Visualiser la distribution des classes et détecter le déséquilibre
3. Analyser les caractéristiques textuelles (longueur, vocabulaire, doublons)
4. Identifier les problèmes à traiter en preprocessing
5. Formuler des hypothèses sur les features discriminantes

**Dataset** : `data/raw/flipkart_data.csv`  
- ~ 10 000 avis produits en anglais  
- Colonnes : `review` (texte), `rating` (int 1-5)

## 0. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '..')  # Accès aux modules src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.float_format', '{:.4f}'.format)

# Palette couleurs projet
PALETTE = {'positive': '#1D9E75', 'neutral': '#888780', 'negative': '#E24B4A'}

print('✅ Imports OK')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')

## 1. Chargement du Dataset

In [ ]:
df = pd.read_csv('../data/raw/flipkart_data.csv')

print(f'Shape         : {df.shape}')
print(f'Colonnes      : {list(df.columns)}')
print(f'Types         :')
print(df.dtypes)
print(f'\nValeurs nulles:')
print(df.isnull().sum())

In [ ]:
# Aperçu des premières lignes
df.head(10)

In [ ]:
# Quelques avis par rating
for rating in [1, 3, 5]:
    ex = df[df['rating'] == rating]['review'].iloc[0]
    print(f'\n--- Rating {rating}★ ---')
    print(ex[:200])

## 2. Distribution des Ratings & Déséquilibre

In [ ]:
# Mapping 3 classes
SENTIMENT_MAP = {1: 'negative', 2: 'negative', 3: 'neutral', 4: 'positive', 5: 'positive'}
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}

df['sentiment'] = df['rating'].map(SENTIMENT_MAP)
df['label'] = df['sentiment'].map(LABEL2ID)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Distribution des Classes — Dataset Flipkart', fontsize=14, fontweight='bold', y=1.02)

# ── Graphique 1 : Ratings bruts ──────────────────────────────────────────
rc = df['rating'].value_counts().sort_index()
bar_colors = ['#E24B4A', '#EF9F27', '#888780', '#5DCAA5', '#1D9E75']
bars = axes[0].bar(rc.index, rc.values, color=bar_colors, edgecolor='white', linewidth=1.5, width=0.6)
axes[0].set_title('Ratings bruts (1 → 5)', fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Nombre d\'avis')
for bar, (k, v) in zip(bars, rc.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 40,
                 f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9, fontweight='bold')

# ── Graphique 2 : Sentiment 3 classes ────────────────────────────────────
sc = df['sentiment'].value_counts().reindex(['positive', 'neutral', 'negative'])
colors_pie = [PALETTE[s] for s in sc.index]
wedges, texts, autotexts = axes[1].pie(
    sc.values, labels=None, colors=colors_pie,
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2.5},
    pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
handles = [mpatches.Patch(color=PALETTE[s], label=f'{s} ({v} avis)') for s, v in sc.items()]
axes[1].legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, -0.12), ncol=1, fontsize=10)
axes[1].set_title('Sentiment 3 classes (après mapping)', fontweight='bold')

# ── Graphique 3 : Ratio déséquilibre ────────────────────────────────────
ratios = {
    'pos/neg': sc['positive'] / sc['negative'],
    'pos/neu': sc['positive'] / sc['neutral'],
    'neu/neg': sc['neutral'] / sc['negative'],
}
bars2 = axes[2].bar(ratios.keys(), ratios.values(),
                    color=['#E24B4A', '#EF9F27', '#888780'], width=0.5,
                    edgecolor='white', linewidth=1.5)
axes[2].axhline(y=1.0, color='green', linestyle='--', linewidth=1.5, label='Équilibre parfait (1.0x)')
axes[2].axhline(y=3.0, color='orange', linestyle=':', linewidth=1.5, label='Seuil alerte (3.0x)')
axes[2].set_title('Ratios de déséquilibre', fontweight='bold')
axes[2].set_ylabel('Ratio')
axes[2].legend(fontsize=9)
for bar, (k, v) in zip(bars2, ratios.items()):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + 0.1,
                 f'{v:.1f}x', ha='center', fontsize=11, fontweight='bold',
                 color='#E24B4A' if v > 3 else 'black')

plt.tight_layout()
plt.savefig('../data/models/plots/01_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'\n⚠️  DÉSÉQUILIBRE SÉVÈRE détecté:')
print(f'   positive/negative = {ratios["pos/neg"]:.1f}x → SMOTE / class_weight OBLIGATOIRE')
print(f'   Si ignoré : le modèle prédit "positive" tout le temps → accuracy=81% mais F1-macro~0.30')

## 3. Analyse de la Qualité des Données

In [ ]:
# ── Doublons ─────────────────────────────────────────────────────────────
n_dup = df.duplicated(subset=['review']).sum()
print(f'Doublons exacts (même texte) : {n_dup} ({n_dup/len(df)*100:.1f}%)')
print()

# Montrer quelques doublons
dup_mask = df.duplicated(subset=['review'], keep=False)
dup_examples = df[dup_mask].sort_values('review').head(6)
print('Exemples de doublons:')
dup_examples[['review', 'rating', 'sentiment']].head(6)

In [ ]:
# ── Longueur des avis ─────────────────────────────────────────────────────
df['word_count'] = df['review'].str.split().str.len()
df['char_count'] = df['review'].str.len()

print('Statistiques longueur (mots):')
print(df.groupby('sentiment')['word_count'].describe().round(2))

# Avis très courts
short = df[df['word_count'] < 5]
print(f'\nAvis très courts (< 5 mots): {len(short)} ({len(short)/len(df)*100:.1f}%)')
print(short[['review', 'rating', 'sentiment']].head(10))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Analyse des Caractéristiques Textuelles', fontsize=14, fontweight='bold')

# ── Distribution longueur par sentiment ──────────────────────────────────
for sentiment in ['positive', 'neutral', 'negative']:
    subset = df[df['sentiment'] == sentiment]['word_count']
    axes[0, 0].hist(subset, bins=40, alpha=0.65, color=PALETTE[sentiment],
                    label=f'{sentiment} ({len(subset)})', density=True)
axes[0, 0].set_title('Distribution longueur (mots) par sentiment', fontweight='bold')
axes[0, 0].set_xlabel('Nombre de mots')
axes[0, 0].set_ylabel('Densité')
axes[0, 0].legend()
axes[0, 0].axvline(x=5, color='red', linestyle='--', alpha=0.5, label='Seuil 5 mots')

# ── Boxplot longueur ────────────────────────────────────────────────────
data_box = [df[df['sentiment'] == s]['word_count'].values
            for s in ['negative', 'neutral', 'positive']]
bp = axes[0, 1].boxplot(data_box, labels=['negative', 'neutral', 'positive'],
                         patch_artist=True, medianprops={'color': 'white', 'linewidth': 2})
for patch, color in zip(bp['boxes'], [PALETTE['negative'], PALETTE['neutral'], PALETTE['positive']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
axes[0, 1].set_title('Boxplot longueur par sentiment', fontweight='bold')
axes[0, 1].set_ylabel('Nombre de mots')

# ── Barplot longueur moyenne par rating ───────────────────────────────────
corr_data = df.groupby('rating')['word_count'].mean()
axes[1, 0].bar(corr_data.index, corr_data.values,
               color=bar_colors, edgecolor='white', linewidth=1, width=0.6)
axes[1, 0].set_title('Longueur moyenne par rating', fontweight='bold')
axes[1, 0].set_xlabel('Rating')
axes[1, 0].set_ylabel('Mots moyens')
for x, y in zip(corr_data.index, corr_data.values):
    axes[1, 0].text(x, y + 0.2, f'{y:.1f}', ha='center', fontweight='bold')

# ── Distribution avis courts ─────────────────────────────────────────────
thresholds = [3, 5, 10, 20, 50]
short_counts = [len(df[df['word_count'] < t]) for t in thresholds]
bars3 = axes[1, 1].bar([f'< {t}' for t in thresholds], short_counts,
                        color='#3B8BD4', edgecolor='white', linewidth=1)
axes[1, 1].set_title('Avis très courts par seuil', fontweight='bold')
axes[1, 1].set_xlabel('Seuil (mots)')
axes[1, 1].set_ylabel('Nombre d\'avis')
for bar, v in zip(bars3, short_counts):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, v + 5,
                    f'{v}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/models/plots/01b_text_stats.png', bbox_inches='tight', dpi=150)
plt.show()

## 4. Analyse du Vocabulaire

In [ ]:
import re

def simple_tokenize(text):
    if not isinstance(text, str):
        return []
    # Supprimer READ MORE et ponctuation
    text = re.sub(r'READ MORE', '', text, flags=re.IGNORECASE)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return [w.lower() for w in text.split() if len(w) >= 3]

# Vocabulaire global
all_tokens = [t for review in df['review'] for t in simple_tokenize(review)]
vocab = Counter(all_tokens)

print(f'Tokens totaux        : {len(all_tokens):,}')
print(f'Vocabulaire unique   : {len(vocab):,}')
print(f'Hapax legomena (1 occurrence) : {sum(1 for c in vocab.values() if c == 1):,}')

print(f'\nTop 20 mots les plus fréquents:')
print(pd.DataFrame(vocab.most_common(20), columns=['mot', 'freq']).to_string(index=False))

In [ ]:
# Vocabulaire spécifique par sentiment
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)
STOP = set(stopwords.words('english'))

def top_words_by_sentiment(sentiment, n=20):
    tokens = [
        t for review in df[df['sentiment'] == sentiment]['review']
        for t in simple_tokenize(str(review))
        if t not in STOP and len(t) >= 4
    ]
    return Counter(tokens).most_common(n)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Top 15 mots significatifs par sentiment (stopwords exclus)', fontsize=13, fontweight='bold')

for ax, sentiment in zip(axes, ['positive', 'neutral', 'negative']):
    top = top_words_by_sentiment(sentiment, 15)
    words, counts = zip(*top)
    bars = ax.barh(list(words)[::-1], list(counts)[::-1],
                   color=PALETTE[sentiment], alpha=0.85, edgecolor='white')
    ax.set_title(f'{sentiment.upper()} ({df[df["sentiment"]==sentiment].shape[0]} avis)',
                 fontweight='bold', color=PALETTE[sentiment])
    ax.set_xlabel('Fréquence')
    for bar, count in zip(bars, list(counts)[::-1]):
        ax.text(count + 1, bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/models/plots/01c_top_words.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. WordClouds

In [ ]:
try:
    from wordcloud import WordCloud

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    fig.suptitle('WordCloud — Mots les plus fréquents par sentiment', fontsize=14, fontweight='bold')

    cmaps = {'positive': 'Greens', 'neutral': 'Greys', 'negative': 'Reds'}
    for ax, (sentiment, cmap) in zip(axes, cmaps.items()):
        texts = ' '.join(
            w for review in df[df['sentiment'] == sentiment]['review'].dropna()
            for w in simple_tokenize(str(review))
            if w not in STOP
        )
        wc = WordCloud(
            width=700, height=450,
            background_color='white',
            colormap=cmap,
            max_words=150,
            collocations=True,
            min_font_size=8
        ).generate(texts)
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        n = (df['sentiment'] == sentiment).sum()
        ax.set_title(f'{sentiment.upper()}\n({n} avis — {n/len(df)*100:.1f}%)',
                     fontsize=13, fontweight='bold', color=PALETTE[sentiment])

    plt.tight_layout()
    plt.savefig('../data/models/plots/01d_wordclouds.png', bbox_inches='tight', dpi=150)
    plt.show()
except ImportError:
    print('wordcloud non installé. Exécuter: pip install wordcloud')

## 6. Signaux Lexicaux (Features d'ingénierie)

In [ ]:
# Ces features seront ajoutées au TF-IDF en Notebook 02
df['has_exclamation'] = df['review'].str.contains('!', regex=False).astype(int)
df['has_question']    = df['review'].str.contains(r'\?', regex=True).astype(int)
df['caps_ratio']      = df['review'].apply(
    lambda t: sum(1 for c in str(t) if c.isupper()) / max(len(str(t)), 1)
)
df['has_read_more']   = df['review'].str.contains('READ MORE', case=False, regex=False).astype(int)
df['has_emoji']       = df['review'].str.contains(
    r'[\U0001F300-\U0001F9FF]', regex=True
).astype(int)

print('Analyse des signaux par sentiment:')
signal_cols = ['has_exclamation', 'has_question', 'caps_ratio', 'has_emoji']
df.groupby('sentiment')[signal_cols].mean().round(4)

In [ ]:
print(f'Avis contenant READ MORE : {df["has_read_more"].sum()} ({df["has_read_more"].mean()*100:.1f}%)')
print('→ Artefact Flipkart à supprimer en preprocessing\n')

# Exemple avec READ MORE
ex = df[df['has_read_more'] == 1]['review'].iloc[0]
print(f'Exemple brut : "{ex[:150]}"')

## 7. Résumé & Décisions pour le Preprocessing

In [ ]:
print('=' * 60)
print('  RÉSUMÉ EDA — DÉCISIONS PREPROCESSING')
print('=' * 60)

issues = [
    ('Déséquilibre classes',    f'{sc["positive"]/sc["negative"]:.1f}x positif/négatif',      '→ SMOTE + class_weight à comparer'),
    ('Doublons',                f'{df.duplicated("review").sum()} doublons exactes ({df.duplicated("review").mean()*100:.1f}%)', '→ Supprimer avant split'),
    ('Avis très courts',        f'{(df["word_count"] < 5).sum()} avis < 5 mots', '→ Filtrer (min_word_len=2 dans TextCleaner)'),
    ('Artefact READ MORE',      f'{df["has_read_more"].sum()} avis affectés', '→ Supprimer via regex'),
    ('Emojis',                  f'{df["has_emoji"].sum()} avis avec emojis', '→ Supprimer (pas de sens en TF-IDF)'),
    ('Longueur moyenne',        f'{df["word_count"].mean():.1f} mots/avis', '→ Normal, pas de troncature nécessaire'),
]

for issue, observation, decision in issues:
    print(f'\n  🔍 {issue}')
    print(f'     Observation : {observation}')
    print(f'     Décision    : {decision}')

print()
print('=' * 60)
print('  ➡️  Suite : Notebook 02 — Preprocessing & Feature Engineering')
print('=' * 60)

In [ ]:
# Sauvegarder le df EDA pour le notebook suivant
import os
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/flipkart_eda.csv', index=False)
print('✅ Sauvegardé: data/processed/flipkart_eda.csv')